# LECTURE 2: FEATURE SELECTION
### *Filter Methods, Wrapper Strategies, Embedded Regularization, and Production Selection Pipelines*
**Course:** Big Data Analytics in Finance (Phân tích Dữ liệu Lớn trong Tài chính)  
**Institution:** School of Applied Mathematics and Economics (SAMI / MFE) - National Economics University (NEU)  
**Instructor:** Dr. Nguyen Tuan Long  
**Reference Slides:** `Slides/05_L02_Feature_selection.pdf`

---

## Executive Summary & Motivation in Quantitative Finance
In quantitative finance and algorithmic trading, quants routinely engineer hundreds or thousands of candidate alpha signals:
- Multi-horizon momentum and moving averages
- Macroeconomic indicators (yield spreads, inflation prints, central bank liquidity)
- Order book microstructure (bid-ask spread, order flow toxicity, tick volume)
- Alternative data (social sentiment, satellite imagery, web traffic)

However, including all candidate features into a machine learning model leads to four severe hazards:
1. **Overfitting & Noise Memorization:** High dimensional spaces ($P \gg N$) allow models to memorize random historical flukes rather than persistent economic laws.
2. **Multicollinearity & Instability:** Highly correlated signals (e.g., 10-day vs. 12-day moving averages) destabilize linear coefficients and inflate variance.
3. **Computational Latency:** High inference latency violates strict financial execution budgets in high-frequency or real-time trading.
4. **Loss of Interpretability:** Black-box models with hundreds of arbitrary features fail internal risk management, investment committees, and financial regulatory audits (e.g., Basel III, SEC/SBV).

**Core Mission:**
> *Feature selection is the principled process of isolating the minimal subset of features that carries maximum predictive power, lowest redundancy, and zero data leakage.*

---


# OVERVIEW: THE FEATURE SELECTION TAXONOMY
---

```
                       +-------------------------------+
                       |   FEATURE SELECTION TAXONOMY  |
                       +---------------+---------------+
                                       |
        +------------------------------+------------------------------+
        |                              |                              |
+-------v-------+              +-------v-------+              +-------v-------+
|    FILTER     |              |    WRAPPER    |              |   EMBEDDED    |
|    METHODS    |              |    METHODS    |              |    METHODS    |
+-------+-------+              +-------+-------+              +-------+-------+
| Model-Agnostic|              | Model-in-Loop |              | In-Algorithm  |
| Fast: O(p)    |              | Slow: O(2^p)  |              | Balanced      |
| Univariate    |              | Interacting   |              | Regularized   |
| (Chi2, ANOVA, |              | (RFE, RFECV,  |              | (L1 Lasso,    |
|  Mutual Info) |              |  Forward/Back)|              |  Tree MDI)    |
+---------------+              +---------------+              +---------------+
                                       |
                               +-------v-------+
                               | HYBRID FUNNEL |
                               | Multi-Stage   |
                               +---------------+
```

---


# MODULE 1: STATISTICAL FILTER METHODS (UNIVARIATE SCREENING)
---

Filter methods evaluate the statistical relationship between each individual feature $X_j$ and the target $y$ independently of any predictive machine learning model.

## 1. The Filter Method Decision Matrix
The correct statistical test is determined strictly by the data types of the input feature and target variable:

| Input Feature ($X$) | Target Variable ($y$) | Recommended Filter Scoring Test | Scikit-Learn Function |
| :--- | :--- | :--- | :--- |
| **Categorical** | **Categorical** | Chi-Squared ($\chi^2$) Test, Mutual Information | `chi2`, `mutual_info_classif` |
| **Numerical** | **Categorical** | ANOVA F-test, Mutual Information | `f_classif`, `mutual_info_classif` |
| **Numerical** | **Numerical** | Pearson Correlation, F-Regression, Mutual Info | `f_regression`, `mutual_info_regression` |

---

## 2. Filter Technique 1: Chi-Squared ($\chi^2$) Test
Tests the null hypothesis ($H_0$) of independence between categorical feature $X$ and categorical target $y$.

**Mathematical Formulation:**
$$\chi^2 = \sum_{i=1}^r \sum_{j=1}^c rac{(O_{ij} - E_{ij})^2}{E_{ij}}$$
where $O_{ij}$ is the observed frequency, and $E_{ij}$ is the expected frequency under independence:
$$E_{ij} = rac{R_i 	imes C_j}{N}$$
- **Non-negativity Requirement:** The input feature matrix must be non-negative ($X \ge 0$). Features must be scaled via `MinMaxScaler` prior to computing $\chi^2$.

#### Hand Calculation Drill:
Consider credit loan applicants ($N=100$) categorized by Debt Level (Low/High) and Loan Default ($0$/$1$):
- Observed Table $O$:
  - Low Debt: Default $0 = 40$, Default $1 = 10$ (Row Total $R_1 = 50$)
  - High Debt: Default $0 = 20$, Default $1 = 30$ (Row Total $R_2 = 50$)
  - Column Totals: Non-Default $C_1 = 60$, Default $C_2 = 40$. Total $N = 100$.
- Expected Table $E$:
  - $E_{11} = (50 	imes 60)/100 = 30.0, \quad E_{12} = (50 	imes 40)/100 = 20.0$
  - $E_{21} = (50 	imes 60)/100 = 30.0, \quad E_{22} = (50 	imes 40)/100 = 20.0$
- Chi-Squared Contribution:
  $$\chi^2 = rac{(40-30)^2}{30} + rac{(10-20)^2}{20} + rac{(20-30)^2}{30} + rac{(30-20)^2}{20} = rac{100}{30} + rac{100}{20} + rac{100}{30} + rac{100}{20} = 3.33 + 5.00 + 3.33 + 5.00 = 16.67$$
- With degrees of freedom $df = (2-1)(2-1) = 1$, $\chi^2 = 16.67 \gg 3.84$ ($p < 0.0001$), decisively rejecting independence!


In [ ]:
# 1. Chi-Squared Filter Method Implementation
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# Replicate the toy contingency drill data
X_toy = np.array([
    [0], [0], [0], [0], [0],  # 50 Low Debt (encoded as 0)
    [1], [1], [1], [1], [1]   # 50 High Debt (encoded as 1)
])
# Match ratio: 40 no-default (0), 10 default (1) for Low Debt; 20 no-default, 30 default for High Debt
y_toy = np.array([
    0]*40 + [1]*10 + [0]*20 + [1]*30
)
X_expanded = np.repeat(X_toy, 10, axis=0)

chi2_stat, p_val = chi2(X_expanded, y_toy)
print("=== Chi-Squared Verification ===")
print(f"Calculated Chi2 Statistic : {chi2_stat[0]:.2f} (Matches hand calculation: 16.67)")
print(f"Calculated p-value        : {p_val[0]:.6f} (Strongly statistically significant!)")


## 3. Filter Technique 2: ANOVA F-test (Numerical In $\to$ Categorical Out)
Tests whether the means of numerical feature $X$ differ significantly across classes of target $y$.

**Mathematical Formulation:**
$$F = \frac{\text{MS}_{\text{between}}}{\text{MS}_{\text{within}}} = \frac{\text{SSB} / (k - 1)}{\text{SSW} / (N - k)}$$
where $\text{SSB} = \sum_{j=1}^k n_j (\bar{x}_j - \bar{x})^2$ and $\text{SSW} = \sum_{j=1}^k \sum_{i=1}^{n_j} (x_{ij} - \bar{x}_j)^2$.

#### Hand Calculation Drill:
Consider 2 classes ($k=2$, $N=6$, $n_1=3, n_2=3$):
- Class 0: values $[1, 2, 3] \implies \bar{x}_1 = 2.0$
- Class 1: values $[5, 6, 7] \implies \bar{x}_2 = 6.0$
- Overall mean: $\bar{x} = 4.0$
- Between-group sum of squares:
  $$\text{SSB} = 3(2 - 4)^2 + 3(6 - 4)^2 = 3(4) + 3(4) = 24.0$$
- Within-group sum of squares:
  $$\text{SSW} = [(1-2)^2 + (2-2)^2 + (3-2)^2] + [(5-6)^2 + (6-6)^2 + (7-6)^2] = (1 + 0 + 1) + (1 + 0 + 1) = 4.0$$
- Degrees of freedom: $df_1 = 2 - 1 = 1$, $df_2 = 6 - 2 = 4$.
- F-Statistic:
  $$F = \frac{24.0 / 1}{4.0 / 4} = \frac{24.0}{1.0} = 24.00$$


In [ ]:
# 2. ANOVA F-test Verification
from sklearn.feature_selection import f_classif

X_anova = np.array([[1], [2], [3], [5], [6], [7]])
y_anova = np.array([0, 0, 0, 1, 1, 1])

F_stat, p_val_anova = f_classif(X_anova, y_anova)
print("=== ANOVA F-test Verification ===")
print(f"Calculated F-statistic : {F_stat[0]:.2f} (Matches hand calculation: 24.00)")
print(f"Calculated p-value     : {p_val_anova[0]:.4f}")


## 4. Mutual Information (Non-Linear Dependency Screening)
Unlike Pearson correlation or ANOVA (which measure purely linear separations), **Mutual Information (MI)** quantifies both linear and complex non-linear relationships.

**Mathematical Formulation:**
$$I(X; Y) = H(X) + H(Y) - H(X, Y) = \iint p(x, y) \log \frac{p(x, y)}{p(x)p(y)} \, dx \, dy$$
- $I(X; Y) = 0$ if and only if $X$ and $Y$ are strictly independent.
- Capable of detecting circular, quadratic ($Y = X^2$), or periodic regimes where Pearson $r \approx 0$.

---

## 5. Limitations of Univariate Screening: Collinearity and Interactions
While filter methods run in ultra-fast $O(p)$ time, they suffer from two fatal blindspots:
1. **Blind to Redundancy:** If Feature 1 and Feature 2 are identical ($X_1 = X_2$), a filter method ranks both at the top, leaving the model burdened with severe multicollinearity.
2. **Blind to Feature Interactions (The XOR Problem):** If $Y = X_1 \oplus X_2$, individually neither $X_1$ nor $X_2$ has any correlation with $Y$ (both get $F=0, \chi^2=0$), and both are mistakenly dropped by univariate screening!


In [ ]:
# 3. Hands-on Lab 1 & 2: Clinical Screening Benchmarks
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# LAB 1: Chi-Squared on Breast Cancer
cancer = load_breast_cancer()
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    cancer.data, cancer.target, test_size=0.3, random_state=42
)

# MinMaxScaler ensures non-negativity for chi2
scaler = MinMaxScaler()
X_tr_c_s = scaler.fit_transform(X_tr_c)
X_te_c_s = scaler.transform(X_te_c)

selector_chi2 = SelectKBest(score_func=chi2, k=4)
X_tr_c_sel = selector_chi2.fit_transform(X_tr_c_s, y_tr_c)
X_te_c_sel = selector_chi2.transform(X_te_c_s)

clf_4f = LogisticRegression(random_state=42).fit(X_tr_c_sel, y_tr_c)
acc_4f = clf_4f.score(X_te_c_sel, y_te_c)

print("=== Hands-on Lab 1: Chi-Squared Selection ===")
selected_indices = selector_chi2.get_support(indices=True)
print("Selected Top 4 Features:", [cancer.feature_names[i] for i in selected_indices])
print(f"Accuracy with only 4 features (reduced from 30): {acc_4f:.4f}")


# MODULE 2: WRAPPER METHODS & SEARCH STRATEGIES
---

Wrapper methods treat feature selection as an optimization problem where a machine learning model is evaluated in-the-loop on candidate feature subsets.

## 1. Search Strategies
Evaluating all $2^p - 1$ subsets is computationally impossible ($O(2^p)$ combinatorial explosion for $p > 30$). Heuristic search algorithms are utilized:
- **Forward Selection:** Starts with zero features; greedily adds the single feature that most improves model score ($O(p^2)$ models).
- **Backward Elimination:** Starts with all $p$ features; iteratively removes the feature whose removal degrades performance least.
- **Recursive Feature Elimination (RFE):** A backward greedy algorithm that uses model weights ($|w_j|$) or impurity importances to prune features iteratively.

---

## 2. Recursive Feature Elimination (RFE) & RFECV
**The RFE Algorithm:**
1. Train estimator on current feature set $S$.
2. Rank features by importance $|w_j|$ (or tree impurity).
3. Prune the lowest ranking feature(s).
4. Repeat until desired subset size $k$ is reached.

**RFECV (Cross-Validated RFE):**
Instead of manually guessing $k$, RFECV evaluates out-of-fold cross-validation scores across all subset sizes $k \in [1, p]$ and automatically picks the optimal subset size $k^*$.

> **Big Data Systems Note (Distributed Execution & Memory Management):**
> When executing recursive wrapper algorithms like `RFECV` on large distributed clusters (PySpark, Ray, or Databricks), configure `n_jobs=-1` to utilize all worker cores and set data partitions to $2\times$ to $4\times$ the total cluster core count. This ensures balanced thread scheduling while guarding worker nodes against out-of-memory (OOM) crashes.


In [ ]:
# 4. Hands-on Lab 3: Tuning Optimal Subsets via RFECV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.datasets import make_classification

# 1. Synthesize benchmark: 800 samples, 20 features (5 true signals, 15 pure noise)
np.random.seed(42)
X_wrap, y_wrap = make_classification(
    n_samples=800, n_features=20, n_informative=5, n_redundant=0,
    n_repeated=0, random_state=42
)

# 2. Configure RFECV with 5-Fold Stratified CV
rf_wrap = RandomForestClassifier(n_estimators=50, random_state=42)
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rfecv = RFECV(estimator=rf_wrap, step=1, cv=cv_strat, scoring="accuracy", n_jobs=-1)
rfecv.fit(X_wrap, y_wrap)

print("=== Hands-on Lab 3: RFECV Results ===")
print(f"Optimal Feature Count k* : {rfecv.n_features_}")
print("True Informative Features: First 5 columns (Indices 0-4)")
print("Optimal Selection Mask   :", rfecv.support_[:8], "...")
print(f"Peak CV Accuracy Achieved: {np.max(rfecv.cv_results_['mean_test_score']):.4f}")


## 3. Production Guardrail: Selection Leakage inside Cross-Validation
> **Critical Production Rule:** *Never perform wrapper or filter feature selection on the full dataset before passing it into Cross-Validation!*

If you select features using all samples:
1. The selection step peeks at the validation fold's labels.
2. The CV score becomes artificially inflated and overly optimistic.
3. The model fails disastrously when deployed into live financial production.
*Solution:* Always nest feature selectors inside a Scikit-Learn `Pipeline`.


# MODULE 3: EMBEDDED METHODS & REGULARIZED IMPORTANCE
---

Embedded methods perform feature selection naturally *during* the training of the algorithm. They capture feature interactions like wrapper methods while remaining computationally fast like filter methods.

## 1. Regularization Geometry: $L_1$ Lasso vs. $L_2$ Ridge
- **Lasso ($L_1$ Penalty):** Objective: $\min_w \frac{1}{2N} \|y - Xw\|_2^2 + \alpha \|w\|_1$.
- **Ridge ($L_2$ Penalty):** Objective: $\min_w \frac{1}{2N} \|y - Xw\|_2^2 + \alpha \|w\|_2^2$.

### Geometric Explanation of Exact Zero Sparsity
- The $L_1$ constraint region is a **hyper-diamond** (cross-polytope) with sharp pointed corners on the coordinate axes.
- The quadratic loss contours first intersect the constraint boundary at one of these sharp corners, forcing non-informative weights to become **exactly zero ($w_j = 0$)**.
- In contrast, the $L_2$ constraint is a smooth **hyper-sphere**. The loss contours touch smooth curves, shrinking weights towards zero but **never setting them to zero**.


In [ ]:
# 5. L1 Lasso Sparsity Implementation
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# High-dimensional financial classification simulation (100 samples, 40 features)
X_sparse, y_sparse = make_classification(n_samples=100, n_features=40, n_informative=6, random_state=42)

# L1 regularized logistic regression (solver='liblinear')
lasso_selector = SelectFromModel(
    LogisticRegression(penalty="l1", solver="liblinear", C=0.25, random_state=42)
)
lasso_selector.fit(X_sparse, y_sparse)

support = lasso_selector.get_support()
print("=== L1 Lasso Sparsity Results ===")
print(f"Total Candidate Features : {X_sparse.shape[1]}")
print(f"Features Retained (w != 0): {np.sum(support)}")
print(f"Features Pruned   (w == 0): {np.sum(~support)} -> Sparsity enforced!")


## 2. Tree Feature Importance: MDI vs. Permutation Importance
Tree-based models calculate feature importance via **Mean Decrease in Impurity (MDI)**:
$$\Delta i(t) = i(t) - \frac{N_L}{N} i(t_L) - \frac{N_R}{N} i(t_R)$$

### The Fatal Flaw of Tree MDI: High Cardinality Bias
Tree algorithms evaluate every unique value of a feature as a potential split point. Continuous features with many unique values have far more opportunities to achieve accidental impurity reductions on the training set. Therefore:
> *MDI will rank pure random continuous noise higher than true categorical features!*

### The Antidote: Permutation Feature Importance
Permutation importance evaluates the drop in performance on an **out-of-fold test set** when feature $j$'s values are randomly shuffled:
$$I(f_j) = \text{Score}_{\text{baseline}} - \text{Score}_{\text{permuted}(j)}$$
If a feature is pure noise, shuffling does not degrade predictions (importance $\le 0$).


In [ ]:
# 6. Hands-on Lab 4: Tree MDI vs. Permutation Importance
from sklearn.inspection import permutation_importance

# 1. Load clinical data and inject a pure random noise column (Column 30)
np.random.seed(42)
data_bc = load_breast_cancer()
noise_col = np.random.randn(len(data_bc.data), 1)
X_with_noise = np.hstack([data_bc.data, noise_col]) # 31 features total
y_bc = data_bc.target

X_tr_n, X_te_n, y_tr_n, y_te_n = train_test_split(X_with_noise, y_bc, test_size=0.3, random_state=42)

# Fit Random Forest
rf_diag = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr_n, y_tr_n)

# 2. Compare Training MDI vs Out-of-Fold Permutation Importance for Noise Column
mdi_noise = rf_diag.feature_importances_[-1]
perm_res = permutation_importance(rf_diag, X_te_n, y_te_n, n_repeats=10, random_state=42)
perm_noise = perm_res.importances_mean[-1]

print("=== Hands-on Lab 4: MDI vs Permutation Importance ===")
print(f"Noise Column MDI Importance (Train): {mdi_noise:.5f} (Artificially positive!)")
print(f"Noise Column Permutation (Test)   : {perm_noise:.5f} (Correctly detected as <= 0.0 noise!)")


# MODULE 4: ADVANCED PRODUCTION PIPELINES & PARADOXES
---

## 1. Freedman's Paradox: The Illusion of Signal in Pure Noise
**Freedman's Paradox (1983):**
Consider a dataset where $X$ has $N=100$ samples and $P=50$ features drawn from pure independent standard normal noise $N(0, 1)$, and $y$ is also independent pure noise. True relationship: ZERO.

If an analyst screens the data, picks the top 10 features with highest correlation, and fits an Ordinary Least Squares (OLS) regression:
- The screened regression achieves an artificial $R^2 \approx 0.50$!
- Standard regression F-tests and t-tests report false $p$-values $< 0.001$!
- The analyst erroneously concludes they discovered a revolutionary trading alpha!

This mathematical paradox proves why feature screening outside cross-validation is fatal.


In [ ]:
# 7. Simulation Drill: Freedman's Paradox Demonstration
from sklearn.linear_model import LinearRegression
from scipy import stats

np.random.seed(42)
N, P = 100, 50
X_noise = np.random.randn(N, P)
y_noise = np.random.randn(N) # Completely independent noise!

# 1. Univariate screening: pick top 10 features with highest correlation
corrs = [stats.pearsonr(X_noise[:, j], y_noise)[0] for j in range(P)]
top_10 = np.argsort(np.abs(corrs))[-10:]

# 2. Fit regression on screened pure noise features
reg_freedman = LinearRegression().fit(X_noise[:, top_10], y_noise)
r2_fake = reg_freedman.score(X_noise[:, top_10], y_noise)

print("=== Freedman's Paradox Simulation ===")
print("True Underlying Signal: EXACTLY ZERO (Pure Gaussian Noise)")
print(f"Apparent R^2 on screened subset: {r2_fake:.4f} -> Over 40% fake variance explained!")


## 2. Production Solution: Automated Scikit-Learn Pipelines
To eliminate Freedman's Paradox and selection leakage, every feature selection step must be enclosed inside a single `Pipeline`. When `cross_val_score` or `GridSearchCV` is executed, Scikit-Learn re-selects features exclusively on each training fold.

---

## 3. Hands-on Lab 5: Grand 3-Stage Selection Pipeline
In enterprise quantitative systems, a **Multi-Stage Funnel** is the industry standard:
1. **Stage 1 (Unsupervised Filter):** `VarianceThreshold` eliminates constant / zero-variance signals.
2. **Stage 2 (Standardization):** `StandardScaler` prepares features for penalty calculations.
3. **Stage 3 (Embedded Sparsity):** `SelectFromModel(Lasso)` rapidly prunes collinear and non-informative signals.
4. **Stage 4 (High-Capacity Estimator):** A non-linear `GradientBoostingClassifier` trains on the pristine, sparse feature set.


In [ ]:
# 8. Hands-on Lab 5: Grand 3-Stage End-to-End Pipeline
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Complex benchmark dataset: 1000 samples, 50 features (10 informative, 20 redundant, 20 noise)
X_grand, y_grand = make_classification(
    n_samples=1000, n_features=50, n_informative=10, n_redundant=20,
    random_state=42
)
X_g_tr, X_g_te, y_g_tr, y_g_te = train_test_split(
    X_grand, y_grand, test_size=0.25, random_state=42
)

# Build Grand 3-Stage Production Pipeline
grand_pipeline = Pipeline([
    ("zero_var", VarianceThreshold(threshold=0.0)),
    ("scaler", StandardScaler()),
    ("lasso_embed", SelectFromModel(
        LogisticRegression(penalty="l1", solver="liblinear", C=0.25, random_state=42)
    )),
    ("gbdt", GradientBoostingClassifier(n_estimators=100, random_state=42))
])

# Fit on train, evaluate on holdout
grand_pipeline.fit(X_g_tr, y_g_tr)
grand_preds = grand_pipeline.predict(X_g_te)

print("=== Hands-on Lab 5: Grand Pipeline Results ===")
n_selected = np.sum(grand_pipeline.named_steps['lasso_embed'].get_support())
print(f"Features Retained by Embedded Funnel: {n_selected} / 50")
print("\nHoldout Performance Report:")
print(classification_report(y_g_te, grand_preds, digits=4))


---

## 4. Master Feature Selection Decision Matrix & Actionable Checklist
| Feature Selection Paradigm | Computational Complexity | Captures Multicollinearity? | Captures Non-Linear Interactions? | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Filter Methods** | Very Fast: $O(p)$ | No | No (except Mutual Info) | Ultra-high dimension screening ($p > 10,000$). |
| **Wrapper Methods** | Very Slow: $O(k \cdot p)$ | Yes | Yes (dependent on estimator) | Final subset optimization when $p < 50$. |
| **Embedded ($L_1$ Lasso)** | Fast: $O(p \cdot N)$ | Yes (prunes collinear) | No (linear model) | Linear models, high-dimensional tabular data. |
| **Permutation Importance**| Moderate: $O(M \cdot N)$ | Yes | Yes | Model-agnostic truth auditing on holdout test set. |
| **Multi-Stage Funnel** | Optimal | Yes | Yes | Enterprise production: Filter $\to$ Embedded $\to$ Model. |
